# MiniGeo Colab 模板

本模板用于在 Colab Pro 中运行 MiniGeo 的模型服务评测、SFT 数据构建和 QLoRA 配置检查。默认本地仓库已经包含 benchmark、RAG corpus 和 SFT 草案数据。

## 1. 获取代码并安装依赖

在 Colab 中先挂载 Drive 或克隆仓库，然后安装模型依赖。

In [ ]:
# 示例：按你的实际仓库地址修改
# !git clone https://github.com/<your-org>/MiniGeo.git
# %cd MiniGeo
!python -m pip install --upgrade pip
!python -m pip install -r requirements-dev.txt
!python -m pip install -r requirements-model.txt

## 2. 本地数据和配置检查

先重新生成 SFT 草案数据，并检查 QLoRA 配置与路径。

In [ ]:
!python scripts/build_sft_corpus.py
!python scripts/train_lora.py --check-only
!python -m pytest -q

## 3. 启动 OpenAI-compatible Qwen 服务

可以用 vLLM、text-generation-inference 或其他兼容 OpenAI API 的服务。下面是环境变量示例。

In [ ]:
import os
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["MINIGEO_MODEL"] = "Qwen/Qwen3.5-2B"
os.environ["MINIGEO_EMBEDDING_BASE_URL"] = "http://localhost:8000/v1"
os.environ["MINIGEO_EMBEDDING_API_KEY"] = "EMPTY"
os.environ["MINIGEO_EMBEDDING_MODEL"] = "Qwen/Qwen3-Embedding-0.6B"
os.environ["MINIGEO_RERANKER_BASE_URL"] = "http://localhost:8000/v1"
os.environ["MINIGEO_RERANKER_API_KEY"] = "EMPTY"
os.environ["MINIGEO_RERANKER_MODEL"] = "Qwen/Qwen3-Reranker-0.6B"

## 4. 运行模型服务评测

先运行检索服务消融，再运行 RAG、Verifier、SQL 和 Agent demo。

In [ ]:
# 单 A100 分阶段服务消融：只启动 embedding 服务时先跑这一行；换成 reranker 服务后再跑下一行。
!python scripts/evaluate_retrieval_ablation.py --use-embedding-service
# !python scripts/evaluate_retrieval_ablation.py --use-reranker-service
# 如果 embedding 和 reranker 服务可同时访问，也可以使用：!python scripts/evaluate_retrieval_ablation.py --use-services
!python scripts/model_rag_demo.py
!python scripts/evaluate_verifier.py --use-model
!python scripts/evaluate_sql.py --use-model
!python scripts/agent_demo.py

## 5. 生成结果文档

完成评测后生成本地摘要、主结果表和失败案例文档。

In [ ]:
!python scripts/write_local_results.py
!python scripts/write_report_artifacts.py
!python scripts/audit_project.py